In [1]:
from wimpy import wimpy as wp
from Bio import SeqIO
from tqdm import tqdm
import pandas as pd
import numpy as np
import linecache
import time
import tracemalloc

In [2]:
class Benchmark:
    def __init__(self, num_runs=3, return_data=False):
        self.num_runs = num_runs
        self.return_data = return_data

    def test(self, func, *args, **kwargs):
        print("------------------------")
        print(f"Benchmarking {func.__name__} over {self.num_runs} runs:")

        # Benchmark time
        times = []
        for _ in tqdm(range(self.num_runs), desc="Testing runtime"):
            start = time.time()
            _ = func(*args, **kwargs)
            end = time.time()
            times.append(end - start)

        # Benchmark memory
        memories = []
        for _ in tqdm(range(self.num_runs), desc="Testing memory usage"):
            tracemalloc.start()
            _ = func(*args, **kwargs)
            snapshot = tracemalloc.take_snapshot()
            
            top_stats = snapshot.statistics("lineno")
            if top_stats:
                memories.append(top_stats[0].size / 1024)  # in KiB
            else:
                memories.append(0)
            tracemalloc.stop()
        
        print(f"Average runtime : {np.mean(times):.2f} +/- {np.std(times):.2f} seconds")
        print(f"Average peak memory usage : {np.mean(memories):.2f} +/- {np.std(memories):.2f} KiB")

        if self.return_data: 
            return {"time": times, "memory": memories}
        
benchmark = Benchmark(num_runs=3, return_data=False)
core_benchmark = Benchmark(num_runs=5, return_data=True)

### Load minimap assignments

In [3]:
# promoters
PROM_IDX = {"": -1, "hEf1a1": 0, "hPGK": 1, "CMV": 2, "RSV": 3}
THRES = 0.64

minimap_prom_df = pd.read_csv("./minimap_output/alignment_prom.csv")

# Another region of the sequence may contain the sequence of hEf1a1
# Use this split to prioritize assigning to other promoters
# If not found and hEf1a1 is above threshold, assign to hEf1a1

hef1a1_df = minimap_prom_df.copy()
hef1a1_df.loc[hef1a1_df["target name"] != "hEf1a1", "Perc. Align"] = 0.0
hef1a1_assignment = hef1a1_df.loc[
    hef1a1_df.groupby("name")["Perc. Align"].idxmax(),
    ["name", "target name", "Perc. Align"],
].reset_index(drop=True)

others_df = minimap_prom_df.copy()
others_df.loc[hef1a1_df["target name"] == "hEf1a1", "Perc. Align"] = 0.0
others_assignment = others_df.loc[
    others_df.groupby("name")["Perc. Align"].idxmax(),
    ["name", "target name", "Perc. Align"],
].reset_index(drop=True)

minimap_prom_assignment = pd.merge(
    hef1a1_assignment,
    others_assignment,
    how="outer",
    on="name",
    suffixes=[" hef1a1", " others"],
)

minimap_prom_assignment["target name"] = ""

above_threshold = minimap_prom_assignment["Perc. Align hef1a1"] > THRES
minimap_prom_assignment.loc[above_threshold, "target name"] = (
    minimap_prom_assignment.loc[above_threshold, "target name hef1a1"]
)

above_threshold = minimap_prom_assignment["Perc. Align others"] > THRES
minimap_prom_assignment.loc[above_threshold, "target name"] = (
    minimap_prom_assignment.loc[above_threshold, "target name others"]
)

minimap_prom_assignment = minimap_prom_assignment[["name", "target name"]]
minimap_prom_assignment["Assignment"] = minimap_prom_assignment["target name"].map(
    PROM_IDX
)
minimap_prom_assignment.head()

,name,target name,Assignment
0,003fcd6f-f1f6-4023-993d-f3ee418ac3d8,hEf1a1,0
1,0074f70d-cfae-4b73-a7fb-7b3b997583ca,hEf1a1,0
2,00c15f1a-0a61-4198-9640-d4a4d7e59445,hEf1a1,0
3,00d225ef-c0c7-4b0e-ab52-d2965e87db58,hEf1a1,0
4,00d24c7a-3db7-48e5-88d1-52860303ae7c,hEf1a1,0


In [4]:
# binding sites
BS_IDX = {2: 0, 4: 1, 8: 2, 12: 3}
minimap_bs_df = pd.read_csv("./minimap_output/alignment_numBS.csv")

max_idx = minimap_bs_df.groupby("name")["Perc. Align"].idxmax()
minimap_bs_assignment = minimap_bs_df.loc[max_idx].reset_index(drop=True)

# convert target name to index assignment
minimap_bs_assignment = minimap_bs_assignment[["name", "target name"]]
minimap_bs_assignment["Assignment"] = minimap_bs_assignment["target name"].map(BS_IDX)
minimap_bs_assignment.head()

,name,target name,Assignment
0,003fcd6f-f1f6-4023-993d-f3ee418ac3d8,4,1
1,0074f70d-cfae-4b73-a7fb-7b3b997583ca,2,0
2,00c15f1a-0a61-4198-9640-d4a4d7e59445,2,0
3,00d225ef-c0c7-4b0e-ab52-d2965e87db58,8,2
4,00d24c7a-3db7-48e5-88d1-52860303ae7c,8,2


### Alignment with `wimpy`

In [5]:
# reference sequences from input fasta file
with open("./info/ref_sequences.fasta") as ref_fasta_file:
    ref_seqs = {
        record.id: str(record.seq) for record in SeqIO.parse(ref_fasta_file, "fasta")
    }
    
Puro = ref_seqs['Puro']
GFP = ref_seqs['GFP'][-100:]
A4 = ref_seqs['A4'][-50:]
mRuby = ref_seqs['mRuby'][200:300]
BS10_1 = ref_seqs['BS10_1']
promoters_100k = pd.read_excel('./info/100k-Promoters.xlsx')['Sequence'].to_list()


In [6]:
# load all fastq files
q_scores, lengths, seqs, names = wp.fastqall("./fastq/", return_names=True)
names = [n.split()[0] for n in names]

reading fastq files:   0%|          | 0/1 [00:00<?, ?it/s]

In [7]:
# use bowtile to re-index sequences
new_seq, _, _ = wp.bowtile(seqs, Puro, thresh=0.03, verbose=True)
new_seq = np.array(new_seq)

# filter out failed attempts
reads_correct = new_seq[new_seq != '']
name_reads_correct = np.array(names)[new_seq != '']
num_seqs = len(reads_correct)

bowtile progress:   0%|          | 0/3502 [00:00<?, ?it/s]

In [8]:
# benchmark bowtile performance
benchmark.test(wp.bowtile, seqs, Puro, thresh=0.03, verbose=False)

------------------------
Benchmarking bowtile over 3 runs:


Testing memory usage: 100%|██████████| 3/3 [01:03<00:00, 21.03s/it]

Average runtime : 18.60 +/- 0.13 seconds
Average peak memory usage : 36373.80 +/- 0.00 KiB


In [9]:
# aligning promoters using mRuby as landmark
_, positions_mRuby, _ = wp.tilepin_v2(reads_correct, mRuby, thresh=0.03, verbose=True)

pregions_synTF = wp.chophat(
    reads_correct,
    np.zeros_like(positions_mRuby),
    end_positions=positions_mRuby,
)

# assign promoter variants
thresh = 0.03
synTF_prom_match_ratios, _, synTF_prom_conf = wp.viscount(
    pregions_synTF, promoters_100k, thresh=thresh, tile_len=10, verbose=True
)
synTF_prom_variants = np.argmax(synTF_prom_match_ratios, axis=1)
synTF_prom_variants[np.sum(synTF_prom_match_ratios, axis=1) < thresh] = -1


match sequences to reference:   0%|          | 0/3180 [00:00<?, ?it/s]

matching to reference sequences:   0%|          | 0/4 [00:00<?, ?it/s]

In [10]:
# benchmark tilepin, chophat, viscount performance
benchmark.test(wp.tilepin_v2, reads_correct, mRuby, thresh=0.03, verbose=False)
benchmark.test(
    wp.chophat,
    reads_correct,
    np.zeros_like(positions_mRuby),
    end_positions=positions_mRuby,
)

viscount_performance = core_benchmark.test(
    wp.viscount,
    pregions_synTF,
    promoters_100k,
    thresh=thresh,
    tile_len=10,
    verbose=False,
)

------------------------
Benchmarking tilepin_v2 over 3 runs:


Testing memory usage: 100%|██████████| 3/3 [02:34<00:00, 51.65s/it]


Average runtime : 2.82 +/- 0.09 seconds
Average peak memory usage : 585244.31 +/- 0.00 KiB
------------------------
Benchmarking chophat over 3 runs:


Testing memory usage: 100%|██████████| 3/3 [00:00<00:00,  5.93it/s]


Average runtime : 0.16 +/- 0.02 seconds
Average peak memory usage : 8403.88 +/- 0.00 KiB
------------------------
Benchmarking viscount over 5 runs:


Testing memory usage: 100%|██████████| 5/5 [01:57<00:00, 23.43s/it]

Average runtime : 19.32 +/- 0.11 seconds
Average peak memory usage : 99.47 +/- 0.00 KiB


In [11]:
wimpy_prom_assignment = pd.DataFrame(
    {
        "Read Name": name_reads_correct,
        "Assignment": synTF_prom_variants,
        "Perc. Align": np.max(synTF_prom_match_ratios, axis=1),
    }
)
wimpy_prom_assignment.head()

,Read Name,Assignment,Perc. Align
0,ed6336bd-7541-4a4d-83d7-299e17f6675a,1,0.667319
1,e4b61b30-e0c5-42c3-8c16-d724481e3889,2,0.986627
2,eb6c2301-79e0-4756-975c-517d4f38c3ee,2,0.945022
3,9f92c4f6-fbec-425d-b81e-b183581e055e,-1,0.000000
4,664eb670-8be0-4c79-a7fc-4f2e80f05fef,3,0.898113


In [12]:
_, positions_GFP, _ = wp.tilepin_v2(reads_correct, GFP, thresh=0.03, verbose=True)
_, positions_A4, _ = wp.tilepin_v2(reads_correct, A4, thresh=0.03, verbose=True)

p_regions = wp.chophat(
    reads_correct,
    positions=positions_A4,
    end_positions=positions_GFP,
)

nbs, _ = wp.fastar(p_regions, BS10_1, tile_len=6, bw=8)

#Assign anything with 10 or more binding sites to 12, anything between 7 & 10 binding sites to 8, and anything between 4 & 6 to 4. All else go to 0 
nbs[nbs > 9.2] = 12
nbs[(nbs > 6.9) & (nbs < 10)] = 8
nbs[(nbs > 3.9) & (nbs < 6.1)] = 4
nbs[(nbs != 2) & (nbs != 4) & (nbs != 8) & (nbs != 12)] = -1

#Reassign the number of binding sites to 0, 1, 2, 3, and 4
value_map = {-1: -1, 2: 0, 4: 1, 8: 2, 12:3}
bs_variants = [value_map[x] for x in nbs]

match sequences to reference:   0%|          | 0/3180 [00:00<?, ?it/s]

match sequences to reference:   0%|          | 0/3180 [00:00<?, ?it/s]

In [13]:
# benchmark fastar performance
fastar_performance = core_benchmark.test(wp.fastar, p_regions, BS10_1, tile_len=6, bw=8)


------------------------
Benchmarking fastar over 5 runs:


Testing memory usage: 100%|██████████| 5/5 [00:31<00:00,  6.24s/it]

Average runtime : 4.38 +/- 0.04 seconds
Average peak memory usage : 377.47 +/- 0.00 KiB


In [14]:
wimpy_bs_assignment = pd.DataFrame(
    {
        "Read Name": name_reads_correct,
        "Assignment": bs_variants,
        "Perc. Align": np.ones_like(nbs),
    }
)
wimpy_bs_assignment.head()

,Read Name,Assignment,Perc. Align
0,ed6336bd-7541-4a4d-83d7-299e17f6675a,3,1.0
1,e4b61b30-e0c5-42c3-8c16-d724481e3889,1,1.0
2,eb6c2301-79e0-4756-975c-517d4f38c3ee,1,1.0
3,9f92c4f6-fbec-425d-b81e-b183581e055e,-1,1.0
4,664eb670-8be0-4c79-a7fc-4f2e80f05fef,3,1.0


### Comparing results to ground truths

In [15]:
# loading ground truths for promoters
ground_truth_prom_assignment = pd.read_csv("./ground_truth/GroundTruth_prom_assignments.csv")
ground_truth_prom_assignment["Read Name"] = ground_truth_prom_assignment[
    "Read Name"
].apply(lambda x: x.split()[0])

# in ground truth file the promoter indices are different from what we use here
# use this mapping to convert
index_correction = {0: -1, 1: 1, 2: 2, 3: 3, 4: 0}
ground_truth_prom_assignment["Assignment"] = ground_truth_prom_assignment[
    "Assignment"
].map(index_correction)

# ground_truth_prom_assignment.head()

In [16]:
prom_prediction = pd.merge(
    ground_truth_prom_assignment,
    wimpy_prom_assignment,
    on="Read Name",
    suffixes=("", "_WIMPY"),
    how="left",
)

prom_prediction = pd.merge(
    prom_prediction,
    minimap_prom_assignment,
    left_on="Read Name",
    right_on="name",
    suffixes=("", "_MINIMAP"),
    how="left",
).loc[:, ["Read Name", "Assignment", "Assignment_WIMPY", "Assignment_MINIMAP"]]

prom_prediction.fillna(-1, inplace=True)
prom_prediction = prom_prediction[prom_prediction["Assignment"] != -1].copy()


In [17]:
for method in ["WIMPY", "MINIMAP"]:
    prom_prediction[f"Prediction_{method}"] = "Correct"
    prom_prediction.loc[
        prom_prediction[f"Assignment_{method}"] != prom_prediction["Assignment"],
        f"Prediction_{method}",
    ] = "Wrong"
    prom_prediction.loc[
        prom_prediction[f"Assignment_{method}"] == -1, f"Prediction_{method}"
    ] = "Unassigned"

    print("-------------------------")
    print(prom_prediction.groupby(f"Prediction_{method}").size() / len(prom_prediction))
    print()


-------------------------
Prediction_WIMPY
Correct       0.996387
Unassigned    0.002007
Wrong         0.001606
dtype: float64

-------------------------
Prediction_MINIMAP
Correct       0.937375
Unassigned    0.025692
Wrong         0.036933
dtype: float64



In [18]:
# ground truths for binding sites
minP = ref_seqs["minP"]
_, positions_minP, _ = wp.tilepin_v2(reads_correct, minP, thresh=0.03, verbose=True)

bs_lengths = positions_minP - positions_A4
bs_assignment = np.digitize(bs_lengths, bins=[60, 160, 255, 415, 555]) - 1
bs_assignment[bs_assignment > 3] = -1

ground_truth_bs_assignment = pd.DataFrame(
    {
        "Read Name": name_reads_correct,
        "Assignment": bs_assignment,
    }
)

# ground_truth_bs_assignment.head()

match sequences to reference:   0%|          | 0/3180 [00:00<?, ?it/s]

In [19]:
bs_prediction = pd.merge(
    ground_truth_bs_assignment,
    wimpy_bs_assignment,
    on="Read Name",
    suffixes=("", "_WIMPY"),
    how="left",
)

bs_prediction = pd.merge(
    bs_prediction,
    minimap_bs_assignment,
    left_on="Read Name",
    right_on="name",
    suffixes=("", "_MINIMAP"),
    how="left",
).loc[:, ["Read Name", "Assignment", "Assignment_WIMPY", "Assignment_MINIMAP"]]

bs_prediction.fillna(-1, inplace=True)
bs_prediction = bs_prediction[bs_prediction["Assignment"] != -1].copy()


In [20]:
for method in ["WIMPY", "MINIMAP"]:
    bs_prediction[f"Prediction_{method}"] = "Correct"
    bs_prediction.loc[
        bs_prediction[f"Assignment_{method}"] != bs_prediction["Assignment"],
        f"Prediction_{method}",
    ] = "Wrong"
    bs_prediction.loc[
        bs_prediction[f"Assignment_{method}"] == -1, f"Prediction_{method}"
    ] = "Unassigned"

    print("-------------------------")
    print(bs_prediction.groupby(f"Prediction_{method}").size() / len(bs_prediction))
    print()

-------------------------
Prediction_WIMPY
Correct       0.967767
Unassigned    0.024854
Wrong         0.007379
dtype: float64

-------------------------
Prediction_MINIMAP
Correct       0.939417
Unassigned    0.011650
Wrong         0.048932
dtype: float64

